# 🧪 W10-D5 Realization Rollback 与 FrozenExecutionContext

> 配套阅读：同名 `.md`。本 notebook 只用小规模、可重复的模拟来验证核心治理约束。

**实验目标：** 模拟 savepoint 风格的原子落地/恢复、软归档，以及冻结上下文的完整性绑定。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

from copy import deepcopy
from dataclasses import dataclass, field
from hashlib import sha256

records = {"workflow": {"state": "draft"}, "binding": {"active": False}, "prompt": {"current": "v2"}}

def realize(db, fail_at=None):
    outer = {"attempt": 1, "status": "in_progress"}   # 外层记录应保留
    savepoint = deepcopy(db)
    try:
        db["workflow"]["state"] = "published"
        if fail_at == "workflow": raise RuntimeError("workflow spec invalid")
        db["binding"]["active"] = True
        if fail_at == "binding": raise RuntimeError("binding invalid")
        db["prompt"]["current"] = "v3"
        outer["status"] = "completed"
    except RuntimeError as err:
        db.clear(); db.update(savepoint)
        outer.update(status="failed", error=str(err))
    return outer

failed_db = deepcopy(records)
print("失败 attempt:", realize(failed_db, "binding"))
print("失败后的对象（已恢复 savepoint）:", failed_db)


In [ ]:
completed_db = deepcopy(records)
record = realize(completed_db)
print("成功 attempt:", record)
print("已落地对象:", completed_db)

def rollback(db):
    # 语义化撤回：不 DELETE，保留对象与历史。
    db["workflow"]["state"] = "archived"
    db["binding"]["active"] = False
    db["prompt"]["current"] = "v2"  # 版本指针回退，v3 历史仍可审计
    return "rolled_back"

print("回滚状态:", rollback(completed_db))
print("回滚后对象:", completed_db)
assert completed_db["workflow"]["state"] == "archived"
assert completed_db["binding"]["active"] is False
assert completed_db["prompt"]["current"] == "v2"
print("恢复验证通过")


In [ ]:
@dataclass(frozen=True)
class FrozenExecutionContext:
    tenant_id: int
    effect_policy: str
    policy_digest: str
    trace_id: str

policy = "conditional_write|mandatory"
fec = FrozenExecutionContext(7, "conditional_write", sha256(policy.encode()).hexdigest(), "trace-99")
print("FEC:", fec)
try:
    fec.effect_policy = "read_only"
except Exception as err:
    print("原地修改被阻止:", type(err).__name__)

def verify_fec(context, expected_policy):
    return context.policy_digest == sha256(expected_policy.encode()).hexdigest()
print("原策略完整性:", verify_fec(fec, policy))
print("篡改后的策略完整性:", verify_fec(fec, "read_only|none"))


In [ ]:
states = ["draft", "published", "archived"]
plt.figure(figsize=(6.2, 3))
plt.plot(states, [0, 1, 0], marker="o", linewidth=2, color="#4C78A8")
plt.yticks([0, 1], ["inactive", "active"]); plt.title("回滚是状态转换，不是删除")
plt.grid(axis="y", alpha=.25); plt.tight_layout(); plt.show()
